In [1]:
import gaussian_to_extxyz

## Now that training set configurations have been selected and written to individual txt files of the format 'Atom_type x y z', you can use the bash scripts in './bash_scripts' to write Gaussian input files and run your calculations.

### To construct the gaussian input files, you will need three files: gaussian_header.txt which contains the Gaussian calculation information; gaussian_bonding.txt which contains the bonding information for your given molecule/cluster (you can generate this in something like GaussView or skip); and bash_combining.sh which combines the 3 relevant files (including the config coordinates) into a new gaussian input file (named gauss_tag_0.gjf). Adust the file names in bash_combining.sh to your needs.  

## Submitting the jobs:

### To conveniently compute the 1000+ Gaussian jobs on an HPC, you can run them in loops of 10-20 calculations (depending on atom number). While you could submit each calculation independently, that would be impolite to the other HPC users, and each calculation takes ~20-120 min to run. Instead the sbatch script gauss_sub_loop.sbatch runs x calculations in series over 24 hours. With 20 calculations per job, it is easy to run 1000 DFT calculations within 50 computing jobs in 1 day. To launch the jobs using sed to insert the correct calculation indices, use the 'submit_loop.sh' script. NOTE: The sbatch script is currently set to delete the checkpoint file after each calculation completes. This is because of storage space on the HPC, as these chk files can get larger. If you do not have this issue, you can comment it out. 



# Post-Gaussian calculations:

## To convert from a Gaussian output into an extended xyz file for learning the forces and energies or the dipole moments, we need the positions, forces,  'standard' positions, energy, and dipole moment:

### Remember that the Gaussian-calculated forces are valid with respect to the 'input' orientation. The dipole moment is valid w.r.t. the Gaussian-rotated 'standard' orientation. You will need 2 different xyz files when training the different models, containing different types of positions. 

### First use bash scripting to extract the info from each .log file (Could be done in python too, but it's very easy with awk).

#### We can do this all at once with the command:

`for i in $(seq 0 1000); do awk '/Input orientation:/{{getline;getline;getline;getline;} for(i;i<=28;i++) {getline;print $2 " " $4 " " $5 " " $6}}' gauss_tag_$i.log >> ../positions_abc.txt; awk '/Forces \(Hartrees/{{getline;getline;} for(i;i<=28;i++) {getline;print $3 " " $4 " " $5}}' gauss_tag_$i.log >> ../forces_abc.txt; awk '/SCF Done/{print $5}' gauss_tag_$i.log >> ../energies_abc.txt; awk  '/Dipole moment/{getline;print $2 " " $4 " " $6}'  gauss_tag_$i.log >> ../dipole_abc.txt; awk '/Charges from ESP fit,/{{getline;getline;} for (i;i<=28;i++) {getline;print $0}}' gauss_tag_$i.log >> ../charges_abc.txt;  awk '/Standard orientation:/{{getline;getline;getline;getline;} for(i;i<=28;i++) {getline;print $2 " " $4 " " $5 " " $6}}' gauss_tag_$i.log >> ../positions_standard_abc.txt; done`

#### You can generate this bash command with replaced values for file names using the function 'gaussian_bash_parsing(number_files, number_atoms, config_tag, output_tag)' where number_files is the number of gaussian log files, config_tag is the string naming the gaussian calculations (i.e. the tag is 'py1w_5Kbins' for gauss_py1w_5Kbins_0.log), and output_tag is the string naming each of the output txt files. The function by default will include the bash for all of the properties above (positions (pos), forces, energies, dipoles, charges, and the standard oriented positions (pos_standard)), but if you don't want one of them, you can set it to false in the function call (i.e. gaussian_bash_parsing(1000, 29, py1w_bins, 1Kbins_py1w, charges=False))
 
### Example: 


In [5]:
bash_command = gaussian_to_extxyz.gaussian_bash_parsing(1000, 41, 'prl3w_21opp_rand', 'opprand_prl3w')

for i in $(seq 0 999); do awk '/Input orientation:/{{getline;getline;getline;getline;} for(i;i<=40;i++) {getline;print $2 " " $4 " " $5 " " $6}}' gauss_prl3w_21opp_rand_$i.log >> ../positions_opprand_prl3w.txt; awk '/Forces \(Hartrees/{{getline;getline;} for(i;i<=40;i++) {getline;print $3 " " $4 " " $5}}' gauss_prl3w_21opp_rand_$i.log >> ../forces_opprand_prl3w.txt; awk '/SCF Done/{print $5}' gauss_prl3w_21opp_rand_$i.log >> ../energies_opprand_prl3w.txt; awk  '/Dipole moment/{getline;print $2 " " $4 " " $6}'  gauss_prl3w_21opp_rand_$i.log >> ../dipole_opprand_prl3w.txt; awk '/Charges from ESP fit,/{{getline;getline;} for (i;i<=40;i++) {getline;print $0}}' gauss_prl3w_21opp_rand_$i.log >> ../charges_opprand_prl3w.txt;  awk '/Standard orientation:/{{getline;getline;getline;getline;} for(i;i<=40;i++) {getline;print $2 " " $4 " " $5 " " $6}}' gauss_prl3w_21opp_rand_$i.log >> ../positions_standard_opprand_prl3w.txt; done


### If calculated, we can also get polarizability, but this isn't currently implemented in the parsing function:

`for i in $(seq 0 99); do awk '/Dipole polarizability\, Alpha \(input orientation/{{getline;getline;getline;getline;getline} for (i;i<=5;i++) {get
line;print $1 " " $3}}' gauss_abc_disp_$i.log >> polar_abc.txt; done`

#### The polarizability from above is written in scientific notation with D rather than E. Numpy can't read this, so use sed to replace all D with E:
`sed -i -e 's/D/E/g' polar_abc.txt`

## Once you have your parsed textfiles, make sure they're all in one folder, and you can use the remaining functions in gaussian_to_extxyz.py to create the extxyz files for training. 

#### First, to make the forces and energies model xyz files, use the function load_forces_energy_data(num_files, num_atoms, output_tag, filepath=str) which returns a dataframe and a vector of energies. Then use write_energy_forces_xyz(coords_forces_data, energy_data, output_filename=str, num_configs=int, num_atoms=int) with the previous function's output to write the xyz file.

### Example:

In [2]:
coords_forces, energy = gaussian_to_extxyz.load_forces_energy_data(1000, 29, 'rand1000py1w', filepath='/Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/pyr_1w/gauss_outs/')

      id         x         y         z        fx        fy        fz
0      C  3.525579  0.000000 -0.010172 -3.117872 -0.549344 -0.345638
1      C  2.818068 -1.202562 -0.071303 -1.074924  3.347797  0.607022
2      C  1.410002 -1.214619  0.021054  0.390935 -0.963599 -0.597551
3      C  0.682341 -0.011622  0.017560  0.215965  0.513326  0.291199
4      C  1.405942  1.199197  0.041075  0.707188  1.463996 -0.244568
...   ..       ...       ...       ...       ...       ...       ...
28995  H -4.581842  0.018115 -0.086353 -1.615852 -0.227667  0.090826
28996  H -3.447221  2.111615 -0.064101 -0.609746  1.649059 -0.085916
28997  O -2.271607  0.617029  3.075290  0.001426 -0.320012 -0.726982
28998  H -2.590342  0.528409  2.152330  0.100646  0.223539  0.792574
28999  H -1.386583  0.248978  3.004476 -0.135343  0.121216  0.069974

[29000 rows x 7 columns]
Number of configurations loaded correctly! 1000 configurations.


In [3]:
gaussian_to_extxyz.write_energy_forces_xyz(coords_forces, energy, output_filename='/Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/pyr_1w/configs_rand1000py1w_TEST.xyz', num_configs=1000, num_atoms=29)

Wrote 1000 configurations to /Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/pyr_1w/configs_rand1000py1w_TEST.xyz


#### To generate the file for training dipoles, the process is similar. Use the load_dipoles_data(num_files, num_atoms, output_tag, filepath=str) function which returns a positions dataframe, dipole matrix, and energy vector. Then use write_dipoles_xyz(coords_data, dipoles_data, energy_data, output_filename=str, num_configs=int, num_atoms=int) to write the file. 

### Example:

In [4]:
coords, dipoles, energy = gaussian_to_extxyz.load_dipoles_data(1000, 29, 'rand1000py1w', filepath='/Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/pyr_1w/gauss_outs/')

Number of configurations loaded correctly! 1000 configurations.


In [5]:
gaussian_to_extxyz.write_dipoles_xyz(coords, dipoles, energy, output_filename='/Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/pyr_1w/configs_rand1000py1w_dipoles_TEST.xyz', num_configs=1000, num_atoms=29)

Wrote 1000 configurations to /Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/pyr_1w/configs_rand1000py1w_dipoles_TEST.xyz


### Next, you can view the constructed xyz files in VMD and move them to a directory for training your ML model. Make sure to do this for training and validation sets! Additional configuration sets can also be used as test sets to test the model at the end of training. 

In [21]:
n_atoms = 38
trajs = 500

coords_forces, energy = gaussian_to_extxyz.load_forces_energy_data(trajs, n_atoms, '4ssrand_fl4w', filepath='/Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/fluor_4w/gauss_outs/')
gaussian_to_extxyz.write_energy_forces_xyz(coords_forces, energy, output_filename='/Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/fluor_4w/configs_4ssrand_fl4w.xyz', num_configs=trajs, num_atoms=n_atoms)

coords, dipoles, energy = gaussian_to_extxyz.load_dipoles_data(trajs, n_atoms, '4ssrand_fl4w', filepath='/Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/fluor_4w/gauss_outs/')
gaussian_to_extxyz.write_dipoles_xyz(coords, dipoles, energy, output_filename='/Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/ml_training/fluor_4w/configs_4ssrand_fl4w_standard_debye.xyz', num_configs=trajs, num_atoms=n_atoms)

      id         x         y         z        fx        fy        fz
0      C -3.666103 -0.722570 -0.066628  0.417979  1.650993 -0.086852
1      C -3.688034  0.703140  0.011444  4.406151 -1.763405  0.215088
2      C -2.468788  1.408151  0.090186 -1.616492 -3.130931 -0.608919
3      C -1.289713  0.665712  0.016728 -0.300340  2.508116  0.378858
4      C -1.282342 -0.724904 -0.013464  0.004868 -1.196014 -0.294033
...   ..       ...       ...       ...       ...       ...       ...
18995  H  1.402137  1.538407 -2.298006 -0.935296 -0.875496 -1.394217
18996  H  0.043424  1.124526 -3.015618 -0.333258  0.247324  0.159192
18997  O -1.759776  1.067708 -3.048157 -0.628688  0.605753  1.929616
18998  H -1.980795  1.081995 -2.068928  0.447735  0.257748 -1.782404
18999  H -1.993943  0.149272 -3.256918  0.297854 -0.201975 -0.341148

[19000 rows x 7 columns]
Number of configurations loaded correctly! 500 configurations.
Wrote 500 configurations to /Users/nale8203/OneDrive - UCB-O365/Documents/2025exps/